In [ ]:
!pip install moleculekit scipy pandas -q

# Moleculekit Protein Interaction Analysis

This notebook analyzes and compares protein-binder interactions (H-bonds, hydrophobic contacts, etc.) for different PDB files.

In [1]:
from moleculekit.molecule import Molecule
import numpy as np
import pandas as pd
from scipy.spatial.distance import cdist

# Determine if running in an environment where we can display output
try:
    from IPython.display import display
except ImportError:
    display = print

In [2]:
def get_atom_coords_and_info(mol, sel):
    """Extracts coordinates and relevant info for a selection."""
    coords = mol.get("coords", sel=sel)
    names = mol.get("name", sel=sel)
    resnames = mol.get("resname", sel=sel)
    resids = mol.get("resid", sel=sel)
    chains = mol.get("chain", sel=sel)
    return coords, names, resnames, resids, chains

def analyze_interactions(mol_path, binder_chain, target_chain):
    """
    Analyzes interactions and returns COUNT and SET of interactions for comparison.
    """
    mol = Molecule(mol_path)
    
    # --- 1. Hydrogen Bonds ---
    sel_binder_polar = f"chain {binder_chain} and name N O S ND1 NE2 SG OH OD1 OD2 OE1 OE2 ND2 NE NZ"
    sel_target_polar = f"chain {target_chain} and name N O S ND1 NE2 SG OH OD1 OD2 OE1 OE2 ND2 NE NZ"
    
    b_coords, b_names, b_resnames, b_resids, _ = get_atom_coords_and_info(mol, sel_binder_polar)
    t_coords, t_names, t_resnames, t_resids, _ = get_atom_coords_and_info(mol, sel_target_polar)
    
    hb_set = set()
    if len(b_coords) > 0 and len(t_coords) > 0:
        dists_hb = cdist(b_coords, t_coords)
        # H-bond Criteria: < 3.5 A between polar heavy atoms
        hb_mask = dists_hb < 3.5
        
        b_idx, t_idx = np.where(hb_mask)
        for bi, ti in zip(b_idx, t_idx):
            # Format: ResNameID(Atom)
            b_info = f"{b_resnames[bi]}{b_resids[bi]}({b_names[bi]})"
            t_info = f"{t_resnames[ti]}{t_resids[ti]}({t_names[ti]})"
            # Store as tuple sorted (or Binder-Target if consistent roles) to ensure uniqueness
            # Since we know binder/target roles, we keep Binder - Target order
            hb_set.add(f"{b_info} - {t_info}")

    # --- 2. Hydrophobic Contacts ---
    hydrophobic_res = "ALA VAL LEU ILE MET PHE TYR TRP PRO"
    sel_binder_hydro = f"chain {binder_chain} and resname {hydrophobic_res} and element C"
    sel_target_hydro = f"chain {target_chain} and resname {hydrophobic_res} and element C"
    
    bh_coords, _, bh_resnames, bh_resids, _ = get_atom_coords_and_info(mol, sel_binder_hydro)
    th_coords, _, th_resnames, th_resids, _ = get_atom_coords_and_info(mol, sel_target_hydro)
    
    hydro_set = set()
    if len(bh_coords) > 0 and len(th_coords) > 0:
        dists_hy = cdist(bh_coords, th_coords)
        # Hydrophobic Criteria: < 4.5 A between Carbon atoms of hydrophobic residues
        hy_mask = dists_hy < 4.5
        
        b_idx, t_idx = np.where(hy_mask)
        for bi, ti in zip(b_idx, t_idx):
            # For hydrophobic, residue-level contact is often more meaningful than atom-level
            # We will store ResNameID - ResNameID pairs
            b_info = f"{bh_resnames[bi]}{bh_resids[bi]}"
            t_info = f"{th_resnames[ti]}{th_resids[ti]}"
            hydro_set.add(f"{b_info} - {t_info}")

    return {
        "File": mol_path,
        "Hbonds_Count": len(hb_set),
        "Hydrophobic_Count": len(hydro_set),
        "Hbond_Set": hb_set,
        "Hydrophobic_Set": hydro_set
    }

def compare_sets(set_a, set_b, label_a, label_b):
    """Compares two sets and returns common and unique items."""
    common = set_a.intersection(set_b)
    unique_a = set_a.difference(set_b)
    unique_b = set_b.difference(set_a)
    return common, unique_a, unique_b

def print_comparison(title, common, unique_a, unique_b, name_a, name_b):
    print(f"\n--- {title} Comparison ---\n")
    print(f"Shared Contacts ({len(common)}):")
    for item in sorted(list(common)): 
        print(f"  [=] {item}")
    
    print(f"\nUnique to {name_a} ({len(unique_a)}):")
    for item in sorted(list(unique_a)):
        print(f"  [A] {item}")
        
    print(f"\nUnique to {name_b} ({len(unique_b)}):")
    for item in sorted(list(unique_b)):
        print(f"  [B] {item}")


In [3]:
# CONFIGURATION
PDB_FILES = [
    {"path": "1IVO.pdb", "binder_chain": "B", "target_chain": "D", "Label": "1IVO"},
    {"path": "modified_color.pdb", "binder_chain": "A", "target_chain": "B", "Label": "Modified"}
]


In [4]:
results = {}

# Run Analysis
for pdb_info in PDB_FILES:
    path = pdb_info["path"]
    label = pdb_info["Label"]
    try:
        print(f"Analyzing {label} ({path})...")
        res = analyze_interactions(path, pdb_info["binder_chain"], pdb_info["target_chain"])
        results[label] = res
    except Exception as e:
        print(f"Error analyzing {path}: {e}")

# Compare if we have exactly 2 results
if len(results) == 2:
    keys = list(results.keys())
    name_a, name_b = keys[0], keys[1]
    res_a, res_b = results[name_a], results[name_b]
    
    # Compare Hydrogen Bonds
    comm_hb, uniq_hb_a, uniq_hb_b = compare_sets(res_a["Hbond_Set"], res_b["Hbond_Set"], name_a, name_b)
    print_comparison("Hydrogen Bonds", comm_hb, uniq_hb_a, uniq_hb_b, name_a, name_b)
    
    # Compare Hydrophobic Contacts
    comm_hy, uniq_hy_a, uniq_hy_b = compare_sets(res_a["Hydrophobic_Set"], res_b["Hydrophobic_Set"], name_a, name_b)
    print_comparison("Hydrophobic Contacts", comm_hy, uniq_hy_a, uniq_hy_b, name_a, name_b)
else:
    print("Need exactly 2 successful analyses to compare.")


Analyzing 1IVO (1IVO.pdb)...
Analyzing Modified (modified_color.pdb)...

--- Hydrogen Bonds Comparison ---

Shared Contacts (0):

Unique to 1IVO (13):
  [A] GLN16(N) - CYS31(O)
  [A] GLN16(O) - ASN32(ND2)
  [A] GLN16(O) - CYS31(O)
  [A] GLN16(O) - CYS33(N)
  [A] GLN384(NE2) - GLN43(O)
  [A] GLN384(OE1) - ARG45(N)
  [A] GLU90(OE1) - LYS28(NZ)
  [A] GLU90(OE2) - LYS28(NZ)
  [A] GLY18(N) - CYS33(O)
  [A] GLY18(O) - ASN32(ND2)
  [A] HIS409(NE2) - ARG45(O)
  [A] HIS409(NE2) - LEU47(N)
  [A] LYS465(NZ) - TRP49(O)

Unique to Modified (15):
  [B] ARG45(N) - GLN384(NE2)
  [B] ARG45(N) - GLN384(OE1)
  [B] CYS31(O) - GLN16(N)
  [B] CYS33(N) - GLN16(O)
  [B] CYS33(O) - GLY18(N)
  [B] GLU25(O) - ASN128(ND2)
  [B] GLU25(OE1) - TYR101(OH)
  [B] GLU51(N) - LYS465(NZ)
  [B] GLU51(OE1) - GLN411(NE2)
  [B] GLY39(N) - ASN12(OD1)
  [B] SER1(N) - TYR101(O)
  [B] TRP50(O) - LYS465(NZ)
  [B] TYR16(OH) - PRO349(N)
  [B] TYR16(OH) - VAL350(N)
  [B] TYR44(OH) - HIS346(ND1)

--- Hydrophobic Contacts Comparison --